# cli

> drive the vault from a terminal

In [ ]:
#| default_exp cli

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

There are no hand-written command wrappers here. `cmd` turns any `Vault` method into a plain
function — `self` and `**kw` stripped — and fastcore's `anno_parser` reads the docments off it, so
every flag, default and help string in the CLI *is* the source of the method it calls. Adding a
method to `CMDS` is the whole of adding a command.

`__delwrap__` is what makes the docments half work — `docments` walks it to find the comments in the
original source, so `anno_parser` and MCP both get help text for free.

In [ ]:
#| export
import json, os, sys
from functools import cache, wraps
from inspect import Parameter, Signature, signature
from fastcore.all import L
from fastcore.script import anno_parser
from vishalakshi.core import Vault
from vishalakshi import acquire, ask, code, extract   # noqa: F401 — these patch the Vault methods the CLI exposes

In [ ]:
#| export
CMDS = ('stats search sections context ask explain read related toc map topic_tree show_topics sources doc document shelves '
        'elsewhere '
        'grab url web crawl arxiv pdf youtube github gh_file add_file add_dir add_tree note code connect forget '
        'apis harvest add_records watch watches unwatch pause poll index_code code_search symbol '
        'where_to_add grep federate drop_shelf categorize categorize_all doctypes of_type ner reshelf '
        'extract '
        'extract_all ask_doc').split()

@cache
def vault(path:str=None) -> Vault:
    'The vault every command shares: `$VISHALAKSHI_VAULT`, else the default path.'
    return Vault(path or os.getenv('VISHALAKSHI_VAULT') or None,
                 offline=bool(os.getenv('VISHALAKSHI_OFFLINE')))

def _spellable(p) -> bool:
    'Can this parameter be written on a command line? `self`, `**kw` and callable defaults cannot.'
    return (p.name != 'self' and p.kind is not Parameter.VAR_KEYWORD
            and not (p.default is not Parameter.empty and callable(p.default)))

def cmd(name:str, vault_fn=vault):
    'A `Vault` method as a plain function: `self`, `**kw` and callable arguments dropped, docments kept.'
    f = getattr(Vault, name)
    g = wraps(f)(lambda **kw: getattr(vault_fn(), name)(**kw))
    # nothing takes a callable default any more — `chat=Chat` was the last, and it is now the
    # plain `chat_kw` dict
    ps = [p for p in signature(f).parameters.values() if _spellable(p)]
    g.__signature__, g.__delwrap__ = Signature(ps), f
    g.__annotations__ = {p.name: p.annotation for p in ps if p.annotation is not Parameter.empty}
    return g

In [ ]:
#| export
def jsonable(o):
    "`json.dumps` fallback: `L` and friends become lists, numpy scalars numbers, the rest strings."
    if hasattr(o, '__iter__') and not isinstance(o, (str, bytes, dict)): return list(o)
    return o.item() if hasattr(o, 'item') else str(o)

def show(r):
    'Print a result: an answer as prose with its citations, anything else as JSON.'
    if isinstance(r, str): return print(r)
    if isinstance(r, dict) and 'answer' in r:
        print(r['answer'])
        # a code citation has no node_id: its handle is the path:line the hit came from
        for c in r.get('cited', []): print(f"  [{c['n']}] {c['breadcrumb']}  {c['node_id'] or c['source'] or ''}")
    else: print(json.dumps(r, indent=2, default=jsonable))

def main():
    'Entry point for the `vishalakshi` command. `vishalakshi <cmd> --help` documents any command.'
    name = sys.argv[1].replace('-', '_') if len(sys.argv) > 1 else None
    if name not in CMDS:
        print(f"Usage: vishalakshi <cmd> [args]\n\n  {'  '.join(CMDS)}\n", file=sys.stderr)
        sys.exit(0 if name is None else 1)
    del sys.argv[1]
    f = cmd(name)
    kw = anno_parser(f, prog=f'vishalakshi {name}').parse_args().__dict__
    show(f(**{k: v for k, v in kw.items() if k not in ('pdb', 'xtra')}))

## Try it

In [ ]:
p = anno_parser(cmd('search'), prog='vishalakshi search')
p.print_help()

usage: vishalakshi find [-h] [--limit LIMIT] [--kind KIND] q

Chunk-level hybrid search (FTS5 + vectors, RRF-fused), each hit carrying its breadcrumb.

positional arguments:
  q              query

options:
  -h, --help     show this help message and exit
  --limit LIMIT  hits to return (default: 10)
  --kind KIND    restrict to one or more KINDS ('note' or 'note,web')


In [ ]:
args = p.parse_args(['late chunking', '--limit', '3', '--kind', 'note,web']).__dict__
test_eq(args['q'], 'late chunking'); test_eq(args['limit'], 3); test_eq(args['kind'], 'note,web')
test_eq(sorted(set(CMDS) - {m for m in dir(Vault) if not m.startswith('_')}), [])   # every command is a real method
test_eq(json.loads(json.dumps(L(['a', 'b']), default=jsonable)), ['a', 'b'])        # `L` is not a list; JSON needs telling
# a callable argument has no command-line spelling, so it is not offered as a flag
_ps = signature(cmd('ask')).parameters
assert 'chat' not in _ps and {'question', 'ref', 'model'} <= set(_ps), list(_ps)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()